# Notebook 30 — Measured latency, memory and full-depth FLOPs (Molmo-7B-D, DeepSeek-VL2-Small)

**Author:** Sunita Sahu · **Revision item:** A5 (iii) latency validation (decision D10) · **GPU required**

Same measurements as notebook 29, for the two models whose released code is not part of transformers:

* **Molmo-7B-D** (`allenai/Molmo-7B-D-0924`, remote code): released processor (`process`), forward with logits at the last position, generation through Molmo's own `generate_from_batch`;
* **DeepSeek-VL2-Small** (`deepseek-ai/deepseek-vl2-small`, the `deepseek_vl2` package pinned to commit `ef9f91e`): released processor (one image, dynamic tiling on), forward and `generate`.

Neither model had the op-level FLOP validation of notebook 27. The CPU dry run of this notebook found two things the full-depth count here will confirm on the released weights:

1. **DeepSeek-VL2's released attention is the absorbed form of multi-head latent attention.** Scores use the 512-wide latent plus the 64-wide rotary key, and values are mixed in the latent (16 heads), instead of the decompressed 16 × 192 / 16 × 128 the cost model assumed. The executed-input prediction here uses the absorbed form; the Table 3 convention is decision D13 in the revision decision log.
2. **Molmo's ViT positional table** was charged as a matmul (+0.19% of its encoder); fixed in the cost model.

Implementation notes recorded with each result: transformers 4.46.3 (required by both code bases); without xformers installed, DeepSeek-VL2's SigLIP attention is routed to PyTorch SDPA (the same attention, a different kernel); Molmo is loaded in bfloat16 from its float32 checkpoint.

### GPU

| GPU (Colab runtime) | Use? | Why |
|---|---|---|
| **A100 (40 GB)** | **Recommended** | Every model fits in bfloat16 (largest: DeepSeek-VL2 at 32 GB, Pixtral at 25 GB); the most common reference GPU for reported latencies |
| H100 (80 GB) | Also good | Faster and more headroom; use it for **both** notebooks if you choose it |
| RTX PRO 6000 / G4 (96 GB) | Works | Newer workstation-class GPU; fine if it is the only large GPU available |
| L4 (24 GB) | No | Pixtral-12B (25 GB of weights) does not fit |
| T4, V100 | No | No bfloat16 support |

**Use one GPU type for notebooks 29 and 30.** Latencies are only comparable on the same GPU; notebook 31 checks this and flags mixed GPUs.

## How to run

1. In Google Drive, create the folder **`MyDrive/ThreadB_GPU`** and upload **`ThreadB_GPU_bundle.zip`** into it (the code; upload it once, every session reuses it).
2. Upload this notebook to the same Drive folder and open it with Colab (opening it from Drive lets Colab autosave the executed copy there). Runtime > Change runtime type > **A100** (see the GPU table), High-RAM if offered. **Start from a fresh runtime** (Runtime > Disconnect and delete runtime) if notebook 29 ran in this session: this notebook needs a different transformers release.
3. Runtime > **Run all**. Allow Drive access when asked.

**Runtime:** about 30–45 minutes on an A100 (about 64 GB of checkpoints, one at a time).

**If the session disconnects:** run all again. Models with a complete result on Drive are skipped; a model interrupted half-way restarts from its first unfinished input size.

**What is saved (all under `MyDrive/ThreadB_GPU/outputs/`):**

| Folder | Contents |
|---|---|
| `results/` | one JSON per model: every raw timing, FLOP count, token account, memory figure and error traceback |
| `logs/` | one log per model, the session transcript (every printed line), crash traces, the pip install log, the executed notebook |
| `tables/` | summary CSVs rebuilt from the JSON files, and the run-status table of each session |
| `environment/` | GPU, driver, CUDA, library versions and `pip freeze` of each session |

The last cell also writes `ThreadB_GPU_outputs_<label>_<time>.zip` with everything above. Notebook 31 analyses the results of notebooks 29 and 30 together.

---
## 1. Settings

In [ ]:
# ---- Settings: edit here, then Runtime > Run all -------------------------------------------------
DRIVE_FOLDER = '/content/drive/MyDrive/ThreadB_GPU'   # holds ThreadB_GPU_bundle.zip; all outputs go under it
BUNDLE_NAME = 'ThreadB_GPU_bundle.zip'
RUN_LABEL = '30_gpu_latency_remote_code'
MODELS_TO_RUN = 'all'            # 'all' (Molmo-7B-D and DeepSeek-VL2-Small), or a list with one of them
IMAGE_SIDES = (448, 1024)        # 1024 x 1024: primary convention (decision D1); 448 x 448: sensitivity check
WARMUP_RUNS = 3                  # untimed passes before each timed series
TIMED_RUNS = 10                  # timed prefill passes; the median and quartiles are reported
DECODE_NEW_TOKENS = 64           # decode cost per token = (T(1 + 64 new tokens) - T(1 new token)) / 64
DECODE_RUNS = 3
COUNT_FULL_DEPTH_FLOPS = True    # op-level FLOP count of the full released model (validates the cost model)
COUNT_ELEMENTWISE = True         # element-wise share of the full-depth FLOPs (reviewer item A5 i)
FORCE_RERUN = False              # True: measure again even when Drive already holds a complete result
DELETE_WEIGHTS_AFTER_EACH_MODEL = True   # frees the Colab disk after each model (weights are re-downloadable)
MINIMUM_GPU_GB = 39

---
## 2. Pinned libraries (legacy stack)

transformers 4.46.3 with its tokenizers, `attrdict3` (the Python 3.10+ fork of the `attrdict` package that DeepSeek-VL2 imports), timm, and the DeepSeek-VL2 package without its dependency pins (it pins torch 2.0.1).

In [ ]:
import subprocess, sys
PINNED_PACKAGES = ['transformers==4.46.3', 'tokenizers>=0.20,<0.21', 'huggingface_hub>=0.23.2,<1.0', 'accelerate>=0.34', 'sentencepiece', 'protobuf', 'einops', 'timm', 'attrdict3', 'packaging', 'pandas', 'scipy']
INSTALL_LOG = ''
def pip_install(arguments):
    """Run pip and keep its output for the install log."""
    global INSTALL_LOG
    completed = subprocess.run([sys.executable, '-m', 'pip', 'install', *arguments], capture_output=True, text=True)
    INSTALL_LOG += f'$ pip install {" ".join(arguments)}\n{completed.stdout}\n{completed.stderr}\n'
    if completed.returncode != 0:
        print(completed.stdout[-3000:], completed.stderr[-3000:])
        raise RuntimeError('pip install failed; see the output above')

pip_install(['--quiet', *PINNED_PACKAGES])
pip_install(['--quiet', '--no-deps', 'git+https://github.com/deepseek-ai/DeepSeek-VL2.git@ef9f91e2b6426536b83294c11742c27be66361b1'])

import transformers
REQUIRED_TRANSFORMERS = '4.46.3'
if transformers.__version__ != REQUIRED_TRANSFORMERS:
    raise RuntimeError(f'transformers {transformers.__version__} is already loaded in this runtime. '
                       'Use Runtime > Restart session (or Disconnect and delete runtime), then Run all again.')
print('transformers', transformers.__version__, '- install log kept for the output folder')

---
## 3. Drive, code bundle, output folders and session transcript

In [ ]:
import os, json, shutil, zipfile, hashlib, traceback
from pathlib import Path
from datetime import datetime

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_FOLDER = Path(DRIVE_FOLDER if IN_COLAB else os.environ.get('THREADB_GPU_FOLDER', Path.cwd() / 'ThreadB_GPU'))
PROJECT_FOLDER.mkdir(parents=True, exist_ok=True)

# Code: the vlm_audit package. Inside the project tree (code_active/notebooks) it is used in place;
# anywhere else it comes from the bundle zip in PROJECT_FOLDER (uploaded once, reused by every session).
if (Path.cwd().parent / 'vlm_audit' / 'latency_measurement.py').exists():
    CODE_DIR, BUNDLE_SHA256 = Path.cwd().parent, None
else:
    bundle_path = PROJECT_FOLDER / BUNDLE_NAME
    if not bundle_path.exists():
        if not IN_COLAB:
            raise FileNotFoundError(f'Put {BUNDLE_NAME} in {PROJECT_FOLDER}')
        from google.colab import files
        print(f'{BUNDLE_NAME} is not in {PROJECT_FOLDER}. Choose it in the upload dialog below.')
        uploaded = files.upload()
        bundle_path.write_bytes(uploaded[next(iter(uploaded))])
    CODE_DIR = Path('/content/threadb_code') if IN_COLAB else PROJECT_FOLDER / 'code'
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    with zipfile.ZipFile(bundle_path) as archive:
        archive.extractall(CODE_DIR)
    BUNDLE_SHA256 = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
sys.path.insert(0, str(CODE_DIR))

OUTPUT_ROOT = PROJECT_FOLDER / 'outputs'
RESULTS_DIR = OUTPUT_ROOT / 'results'          # one JSON per model (raw timings, FLOPs, tokens, errors)
LOG_DIR = OUTPUT_ROOT / 'logs'                  # per-model logs, session transcript, crash traces, pip log
TABLE_DIR = OUTPUT_ROOT / 'tables'              # summary CSVs rebuilt from the JSON results
ENVIRONMENT_DIR = OUTPUT_ROOT / 'environment'   # GPU, driver, library versions, pip freeze per session
for directory in (RESULTS_DIR, LOG_DIR, TABLE_DIR, ENVIRONMENT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
SESSION_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

from vlm_audit.logging_utils import ExperimentLogger, SessionTranscript
if globals().get('transcript') is not None:
    transcript.close()
transcript = SessionTranscript(LOG_DIR, RUN_LABEL)     # every printed line from here on is also on Drive
(LOG_DIR / f'{SESSION_STAMP}_{RUN_LABEL}__pip_install.txt').write_text(INSTALL_LOG, encoding='utf-8')

import vlm_audit.latency_measurement as latency_measurement
print('code      :', latency_measurement.__file__)
print('bundle    :', BUNDLE_SHA256)
print('outputs   :', OUTPUT_ROOT)

---
## 4. GPU check and environment manifest

In [ ]:
import torch
from vlm_audit.latency_measurement import environment_manifest, free_disk_gb

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > A100 (or H100), then Run all.')
properties = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_MEMORY_GB = properties.name, properties.total_memory / 1e9
print(f'GPU: {GPU_NAME}  {GPU_MEMORY_GB:.1f} GB  compute capability {properties.major}.{properties.minor}')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('This GPU has no bfloat16 support (T4, V100). Choose an A100 or H100 runtime.')
if GPU_MEMORY_GB < MINIMUM_GPU_GB:
    print(f'WARNING: {GPU_MEMORY_GB:.0f} GB is below the {MINIMUM_GPU_GB} GB this notebook needs; large models will fail '
          'with out-of-memory errors (recorded in their results).')
print(f'free disk: {free_disk_gb():.0f} GB')

# Hugging Face token: optional (every repository used here is ungated); it only raises rate limits.
# In Colab: key icon on the left > add HF_TOKEN > allow notebook access.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception as error:
    print(f'no HF_TOKEN secret ({type(error).__name__}): anonymous downloads')

SETTINGS_RECORD = {name: globals()[name] for name in (
    'MODELS_TO_RUN', 'IMAGE_SIDES', 'WARMUP_RUNS', 'TIMED_RUNS', 'DECODE_NEW_TOKENS', 'DECODE_RUNS',
    'COUNT_FULL_DEPTH_FLOPS', 'COUNT_ELEMENTWISE', 'FORCE_RERUN', 'DELETE_WEIGHTS_AFTER_EACH_MODEL')}
manifest = environment_manifest()
manifest.update({'run_label': RUN_LABEL, 'bundle_sha256': BUNDLE_SHA256, 'settings': SETTINGS_RECORD})
manifest_path = ENVIRONMENT_DIR / f'{SESSION_STAMP}_{RUN_LABEL}_environment.json'
manifest_path.write_text(json.dumps(manifest, indent=1, default=str), encoding='utf-8')
print('environment manifest:', manifest_path)

---
## 5. Measure both models

Same stages as notebook 29. DeepSeek-VL2-Small holds 32 GB of weights in bfloat16; on a 40 GB A100 its 1024 × 1024 input (about 2,050 tokens) fits, and an out-of-memory error would be recorded, not fatal.

In [ ]:
from vlm_audit.latency_measurement import (MeasurementSettings, build_harness, delete_cached_repository,
                                           load_result, measure_model, release_gpu_memory, save_result,
                                           weights_repository)
import pandas as pd

settings = MeasurementSettings(image_sides=tuple(IMAGE_SIDES), warmup_runs=WARMUP_RUNS, timed_runs=TIMED_RUNS,
                               decode_new_tokens=DECODE_NEW_TOKENS, decode_runs=DECODE_RUNS,
                               count_flops=COUNT_FULL_DEPTH_FLOPS, count_elementwise=COUNT_ELEMENTWISE)
models = list(latency_measurement.REMOTE_CODE_MODELS) if MODELS_TO_RUN == 'all' else list(MODELS_TO_RUN)
print(f'{len(models)} models: {models}')

run_status = []
for model_name in models:
    previous = None if FORCE_RERUN else load_result(RESULTS_DIR, model_name)
    if previous and previous.get('mode') != 'released':
        previous = None                       # ignore dry-run files
    if previous and previous.get('status') == 'complete':
        print(f'{model_name}: complete result already on Drive, skipped (set FORCE_RERUN = True to redo)')
        run_status.append({'model': model_name, 'status': 'complete (earlier session)'})
        continue
    started = datetime.now()
    print(f'\n=== {model_name}   {started:%H:%M:%S}   free disk {free_disk_gb():.0f} GB')
    harness, status, error_text = None, 'failed', ''
    with ExperimentLogger(LOG_DIR, f'{RUN_LABEL}__{model_name}') as logger:
        try:
            logger.section(f'Load {weights_repository(model_name)}')
            harness = build_harness(model_name, mode='released', device='cuda')
            loaded_gb = torch.cuda.memory_allocated() / 1e9
            logger.info(f'loaded in {harness.load_seconds:.0f} s; {loaded_gb:.1f} GB on the GPU; notes: {harness.notes}')
            print(f'  loaded in {harness.load_seconds:.0f} s ({loaded_gb:.1f} GB on the GPU)')
            result = measure_model(harness, settings, logger=logger, previous=previous,
                                   save=lambda partial: save_result(partial, RESULTS_DIR))
            status = result['status']
        except Exception as error:
            error_text = traceback.format_exc()
            logger.error('model run failed', error)
            print(error_text[-3000:])
            record = load_result(RESULTS_DIR, model_name) or {'model': model_name, 'inputs': {}, 'mode': 'released'}
            finished_inputs = [entry for entry in record.get('inputs', {}).values() if entry.get('status') == 'complete']
            record['status'] = 'partial' if finished_inputs else 'failed'
            record['run_error'] = error_text
            save_result(record, RESULTS_DIR)
            status = record['status']
        finally:
            harness = None                        # the harness closures hold the model
            release_gpu_memory()
            if DELETE_WEIGHTS_AFTER_EACH_MODEL:
                freed_gb = delete_cached_repository(weights_repository(model_name))
                print(f'  removed {freed_gb:.1f} GB of cached weights')
            logger.info(f'status: {status}')
    minutes = (datetime.now() - started).total_seconds() / 60
    print(f'  status: {status}   ({minutes:.1f} min)')
    last_error_line = error_text.strip().splitlines()[-1] if error_text else ''
    run_status.append({'model': model_name, 'status': status, 'minutes': round(minutes, 1), 'error': last_error_line})

run_status_table = pd.DataFrame(run_status)
run_status_table.to_csv(TABLE_DIR / f'{SESSION_STAMP}_{RUN_LABEL}_run_status.csv', index=False)
run_status_table

---
## 6. Summary tables and a first look (all results in the Drive folder, notebooks 29 and 30)

In [ ]:
from vlm_audit.latency_analysis import flop_validation_table, latency_table, load_results

all_results = load_results(RESULTS_DIR)          # notebooks 29 and 30 share this folder
summary = latency_table(all_results)
summary.to_csv(TABLE_DIR / 'latency_summary.csv', index=False)
flop_validation = flop_validation_table(all_results)
flop_validation.to_csv(TABLE_DIR / 'flop_validation_full_depth.csv', index=False)
print(f'{len(all_results)} model results; tables written to {TABLE_DIR}')
columns = ['model', 'input', 'status', 'predicted_total_gflops', 'measured_total_gflops', 'prefill_ms_median',
           'vision_ms_median', 'language_ms_median', 'decode_ms_per_token', 'elementwise_share_pct', 'errors']
summary[[column for column in columns if column in summary]].round(2)

In [ ]:
# First look only; the full analysis (proxies, shares, scaling, decode) is notebook 31.
from scipy import stats
if summary.empty:
    print('no results yet')
else:
    if not flop_validation.empty:
        worst = flop_validation.loc[flop_validation.relative_difference_pct.abs().idxmax()]
        print(f'cost model vs full-depth FLOP count: largest |difference| {abs(worst.relative_difference_pct):.3f}% '
              f'({worst.model}, {worst.input}, {worst.component})')
    for input_key, group in summary.groupby('input'):
        usable = group.dropna(subset=['prefill_ms_median', 'predicted_total_gflops'])
        if len(usable) >= 3:
            flops_rho = stats.spearmanr(usable.predicted_total_gflops, usable.prefill_ms_median).statistic
            parameter_rho = stats.spearmanr(usable.parameter_count, usable.prefill_ms_median).statistic
            print(f'{input_key}: n = {len(usable)}; Spearman rho(prefill latency, predicted FLOPs) = {flops_rho:+.2f}; '
                  f'rho(prefill latency, parameter count) = {parameter_rho:+.2f}')
    print('GPUs in these results:', sorted(summary.gpu.dropna().unique()))

---
## 7. Archive every output and close the transcript

In [ ]:
transcript.close()
transcript = None
if IN_COLAB:
    try:
        from google.colab import _message
        executed_notebook = _message.blocking_request('get_ipynb', request='', timeout_sec=120)['ipynb']
        # Colab stores the Google account (display name, user id) in every cell's executionInfo; outputs
        # carry no byline, so drop it before the copy is archived.
        for cell in executed_notebook.get('cells', []):
            cell.get('metadata', {}).get('executionInfo', {}).pop('user', None)
        notebook_copy = LOG_DIR / f'{SESSION_STAMP}_{RUN_LABEL}__executed_notebook.ipynb'
        notebook_copy.write_text(json.dumps(executed_notebook), encoding='utf-8')
        print('executed notebook saved:', notebook_copy)
    except Exception as error:
        print(f'executed notebook not saved automatically ({type(error).__name__}); use File > Download > .ipynb')

archive_path = PROJECT_FOLDER / f'ThreadB_GPU_outputs_{RUN_LABEL}_{SESSION_STAMP}.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(PROJECT_FOLDER))
print(f'archive of every output: {archive_path}  ({archive_path.stat().st_size / 1e6:.1f} MB)')
if IN_COLAB:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted: every file above is saved. Download the archive from Drive, '
          'or keep the outputs folder there for notebook 31.')